**Calculate Correlation Coefficient between station observations and incident pixels**


In [102]:
import xarray
import pandas as pd
import glob
import os
import rasterio
import concurrent.futures
from scipy.stats import spearmanr
import os
import numpy as np
from permetrics.regression import RegressionMetric
import HydroErr as he
import datetime as dt

In [103]:
datasets = ['CHIRPS', 'PERSIANN', 'ERA5L', 'HAR', 'MSWEP'] 

#Read insitu data
df_insitu = pd.read_csv(r'..\Data\Processed\Insitu\insitu_rainfall_1990_2019.csv')
df_insitu['date'] = pd.to_datetime(df_insitu[['year', 'month', 'day']])

#Get path, files and file list for datasets
def extract_date(file, dataset):
    filename = os.path.basename(file)
    date_str = filename[-12:-4]
    return date_str

def get_dataset_info(dataset):
    dataset_path = fr"..\Data\Datasets\{dataset}_prec"
    dataset_tiff_files = glob.glob(os.path.join(dataset_path, "*.tif"))
    dataset_dates = list(map(lambda file: extract_date(file, dataset), dataset_tiff_files))
    return dataset_path, dataset_tiff_files, dataset_dates

dataset_results = {}
for dataset in datasets:
    dataset_path, dataset_tiff_files, dataset_dates = get_dataset_info(dataset)
    dataset_results[f'{dataset}_path'] = dataset_path
    dataset_results[f'{dataset}_files'] = dataset_tiff_files
    dataset_results[f'{dataset}_dates'] = dataset_dates

#dataset_results['MSWEP_files']

In [104]:
#extract station lat and lon in a list 
unique_stn = df_insitu[['index_no','lat','long']].drop_duplicates()
stn_points = list(zip(unique_stn['long'],unique_stn['lat']))
#stn_points
#unique_stn   

In [105]:
def extract_concurrent_grid_data(dataset_results, stn_points, unique_stn):
    df_final = None
    
    for dataset in datasets:
        files = dataset_results[f'{dataset}_files']
        print(f"Processing {dataset} with {len(files)} files...")
        
        def process_file(file):
            date_obj = extract_date(file, dataset) 
            local_records = []
            try:
                with rasterio.open(file) as src:
                    crs = src.crs
                    if crs == "EPSG:4326":
                        sampled_values = list(src.sample(stn_points))
                        for i, value in enumerate(sampled_values):
                            local_records.append({
                                'date': date_obj,
                                'stn_id': unique_stn.iloc[i]['index_no'],
                                'prec_grid': value[0]
                                })
            except Exception as e:
                print(f'Error processing {file}: {e}')
            return local_records

        dataset_records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            results = executor.map(process_file, files)
            for result in results:
                dataset_records.extend(result)
        
        df_dataset = pd.DataFrame(dataset_records)
        
        # CHECK IF EMPTY FIRST
        if not df_dataset.empty:
            # df_dataset = df_dataset[df_dataset['prec_grid'] != -9999.0]
            # df_dataset = df_dataset.dropna().round(1)
            df_dataset.loc[df_dataset['prec_grid'] < 0, 'prec_grid'] = 0
            df_dataset = df_dataset.rename(columns={'prec_grid': f'{dataset}_prec'})
            
            # Merge logic
            if df_final is None:
                df_final = df_dataset
            else:
                df_final = pd.merge(df_final, df_dataset, on=['date', 'stn_id'], how='inner')
                print(f"Merged {dataset}. Resulting shape: {df_final.shape}")
        else:
            print(f"Warning: No data extracted for {dataset}")

    return df_final

df_grid_data = extract_concurrent_grid_data(dataset_results, stn_points, unique_stn)

Processing CHIRPS with 10957 files...
Processing PERSIANN with 10957 files...
Merged PERSIANN. Resulting shape: (942302, 4)
Processing ERA5L with 10956 files...
Merged ERA5L. Resulting shape: (942216, 5)
Processing HAR with 10957 files...
Merged HAR. Resulting shape: (942216, 6)
Processing MSWEP with 10957 files...
Merged MSWEP. Resulting shape: (942216, 7)


In [122]:
#merging station and grid df into a single df
df_insitu1 = df_insitu[['date','index_no','rainfall_sum']]
df_insitu1 = df_insitu1.rename(columns={'rainfall_sum':'prec_insitu', 'index_no':'stn_id'})

df_grid_data['date'] = pd.to_datetime(df_grid_data['date'])
df_insitu1['date'] = pd.to_datetime(df_insitu1['date'])
df_merged = pd.merge(df_grid_data, df_insitu1, on=['date','stn_id'], how='inner')

df_merged.to_csv(r'../Data/Processed/Accuracy_assessment/precipitation_daily_merged.csv', index=False)

**Calculating error metrices & CPI- Daily Scale**

In [125]:
#checkpoint - run from here to reduce code runtime while rerunning
import xarray
import pandas as pd
import glob
import os
import rasterio
import concurrent.futures
from scipy.stats import spearmanr
import os
import numpy as np
from permetrics.regression import RegressionMetric
import HydroErr as he
import datetime as dt

datasets = ['ERA5L', 'HAR', 'CHIRPS', 'MSWEP', 'PERSIANN']
df_merged = pd.read_csv(r'../Data/Processed/Accuracy_assessment/precipitation_daily_merged.csv')
df_merged = df_merged.round(1)
df_merged


,date,stn_id,CHIRPS_prec,PERSIANN_prec,ERA5L_prec,HAR_prec,MSWEP_prec,prec_insitu
0,1990-01-01,102.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1990-01-01,103.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1990-01-01,104.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1990-01-01,105.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1990-01-01,107.0,0.0,0.2,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
905752,2019-12-31,1407.0,0.0,0.0,0.1,0.0,0.0,0.0
905753,2019-12-31,1408.0,0.0,0.0,0.0,0.0,0.0,0.0
905754,2019-12-31,1412.0,0.0,0.0,0.0,0.0,0.0,0.0
905755,2019-12-31,1416.0,0.0,0.0,0.0,0.0,0.0,0.0


In [126]:
#column normalization function
def normalize_column(col):
    return (col - col.min()) / (col.max() - col.min())
          
#calculate accuracy of datasets
def calculate_accuracy(dataframe_prec):
    df_accuracy = pd.DataFrame()

    #datasets = ['ERA5L', 'HARv2', 'CHIRPS', 'MSWEP', 'PERSIANN']

    for dataset in datasets:
        dataset_accuracy = {'dataset': dataset}

        #Convert to array
        prec_grid_array = dataframe_prec[f'{dataset}_prec'].values
        prec_insitu_array = dataframe_prec['prec_insitu'].values
    
        #drop null values, and negative values
        valid_indices = np.logical_and.reduce([
            ~np.isnan(prec_grid_array),
            ~np.isnan(prec_insitu_array),
            prec_grid_array >=  0,
            prec_insitu_array >= 0
        ])
    
        observations = np.sum(valid_indices)
        dataset_accuracy['observations_n'] = observations

        prec_grid_array = prec_grid_array[valid_indices]
        prec_insitu_array = prec_insitu_array[valid_indices]

        #correlation coefficient
        correlation = he.spearman_r(prec_grid_array, prec_insitu_array)
        dataset_accuracy['correlation'] = correlation

        #NRMSE
        nrmse = he.nrmse_mean(prec_grid_array, prec_insitu_array)
        dataset_accuracy['nrmse'] = nrmse

        #Percent bias
        pbias = 100 * np.sum(prec_grid_array - prec_insitu_array) / np.sum(prec_insitu_array)
        dataset_accuracy['pbias'] = pbias

        #Relative standard deviation 
        rsd = (np.std(prec_grid_array))/(np.std(prec_insitu_array))
        dataset_accuracy['rsd'] = rsd

        df_dataset_accuracy = pd.DataFrame([dataset_accuracy])
        df_accuracy = pd.concat([df_accuracy, df_dataset_accuracy], ignore_index=True)

    #Normalization
    df_accuracy['abs_pbias'] = df_accuracy['pbias'].abs()
    df_accuracy['diff_rsd'] = (df_accuracy['rsd']-1).abs()

    df_accuracy_normalized = df_accuracy.copy().drop(columns=['pbias'])

    metrics = ['correlation', 'nrmse', 'abs_pbias', 'diff_rsd']
    for column in metrics:
        df_accuracy_normalized[column] = normalize_column(df_accuracy_normalized[column])

    #converting to scale of 0 to 1 (ideal to worst) - correlation & rsd needs inversion
    df_accuracy_normalized['correlation'] = 1 - df_accuracy_normalized['correlation']

    #get cpi for each (linear)
    df_accuracy_normalized['cpi'] = df_accuracy_normalized['correlation'] + df_accuracy_normalized['nrmse'] + df_accuracy_normalized['abs_pbias'] + df_accuracy_normalized['diff_rsd']
    df_accuracy_normalized['cpi_rank'] = df_accuracy_normalized['cpi'].rank(ascending=True)

    df_accuracy_final = df_accuracy_normalized[['dataset','cpi','cpi_rank']].merge(df_accuracy[['dataset','correlation','nrmse','pbias','rsd', 'observations_n']], on=['dataset'], how='inner')
    return df_accuracy_final

#get daily data accuracy
df_accuracy_daily = calculate_accuracy(df_merged)
df_accuracy_daily

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,ERA5L,0.944119,1.0,0.579087,3.248013,35.633048,1.033310,710697
1,HAR,1.047332,2.0,0.532372,3.357179,16.135607,1.011265,710697
2,CHIRPS,2.061889,4.0,0.279325,3.440636,-11.289981,0.936250,710697
3,MSWEP,2.579338,5.0,0.431975,2.994716,-89.355161,0.140683,710697
4,PERSIANN,1.266717,3.0,0.416585,2.951381,-29.644332,0.573617,710697


**Accuracy and CPI Calculation - Seasonal Scale**

In [127]:
#define seasons
seasons = {
    'winter': [12, 1, 2],
    'pre_monsoon': [3, 4, 5],
    'monsoon': [6, 7, 8, 9],
    'post_monsoon': [10, 11]
}
df_merged1 = df_merged.copy()
df_merged1['date'] = pd.to_datetime(df_merged1['date'])
df_merged1['year'] = df_merged1['date'].dt.year
df_merged1['month'] = df_merged1['date'].dt.month

#create effective year for seasonal analysis
df_merged1['year_seasonal'] = df_merged1['year']
df_merged1.loc[df_merged1['month'] == 12, 'year_seasonal'] = df_merged1.loc[df_merged1['month'] == 12, 'year'] + 1

df_merged1['season'] = (df_merged1['month'].apply(lambda x: [k for k, v in seasons.items() if x in v][0]))

#calculate seasonal rainfall sum for each dataset 
df_seasonal = df_merged1.groupby(by = ['season','year_seasonal'])[['CHIRPS_prec',	'PERSIANN_prec', 'ERA5L_prec', 'prec_insitu','HAR_prec', 'MSWEP_prec']].sum().reset_index()
df_seasonal

,season,year_seasonal,CHIRPS_prec,PERSIANN_prec,ERA5L_prec,prec_insitu,HAR_prec,MSWEP_prec
0,monsoon,1990,107874.7,82539.0,148626.0,83683.9,109828.6,12900.8
1,monsoon,1991,84169.4,76083.1,123149.7,78883.7,104887.9,9060.7
2,monsoon,1992,63587.9,64154.1,97761.2,65389.0,85002.7,8041.3
3,monsoon,1993,85937.5,94388.8,149315.2,79830.4,108237.8,11000.7
4,monsoon,1994,88944.2,77367.0,121043.8,75957.2,97969.4,13897.1
...,...,...,...,...,...,...,...,...
116,winter,2016,2156.7,25781.6,8128.2,1547.3,3679.3,271.5
117,winter,2017,2540.1,1988.2,6087.4,1385.3,4214.2,237.3
118,winter,2018,2585.7,1696.8,5027.6,1146.9,2639.8,149.6
119,winter,2019,8271.2,8354.9,15826.5,7464.1,17361.4,1012.5


In [128]:
#caluclate overall seasonal cpi
df_accuracy_seasonal = (calculate_accuracy(df_seasonal))
df_accuracy_seasonal['season'] = 'all'
#print("OVERALL SEASONAL ACCURACY:")
#print(df_accuracy_seasonal)

#calculate monsoonal cpi
df_monsoon = df_seasonal[df_seasonal['season'] == 'monsoon']
df_accuracy_monsoon = calculate_accuracy(df_monsoon)
df_accuracy_monsoon['season'] = 'monsoon'
#print("MONSOON ACCURACY:")
#print(df_accuracy_monsoon)

#calculate pre-monsoon cpi
df_premonsoon = df_seasonal[df_seasonal['season'] == 'pre_monsoon']
df_accuracy_premonsoon = calculate_accuracy(df_premonsoon)
df_accuracy_premonsoon['season'] = 'premonsoon'
#print("PRE-MONSOON ACCURACY:")
#print(df_accuracy_premonsoon)

#calculate post-monsoon cpi
df_postmonsoon = df_seasonal[df_seasonal['season'] == 'post_monsoon']
df_accuracy_postmonsoon = calculate_accuracy(df_postmonsoon)
df_accuracy_postmonsoon['season'] = 'postmonsoon'
#print("POST-MONSOON ACCURACY:")
#print(df_accuracy_postmonsoon)

#concatenate all cpi
df_accuracy_final = pd.concat([df_accuracy_seasonal, df_accuracy_monsoon, df_accuracy_premonsoon, df_accuracy_postmonsoon])
df_accuracy_final

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n,season
0,ERA5L,2.771946,4.0,0.974841,1.237678,78.362257,1.720465,121,all
1,HAR,1.287309,3.0,0.980728,0.752231,50.496664,1.390935,121,all
2,CHIRPS,1.272154,2.0,0.955528,0.388517,18.599720,1.201758,121,all
3,MSWEP,3.648398,5.0,0.964138,1.414599,-85.913340,0.134880,121,all
4,PERSIANN,1.000000,1.0,0.955142,0.321821,-7.215277,0.857144,121,all
0,ERA5L,2.796546,4.0,0.593771,0.757370,73.775974,2.095938,30,monsoon
1,HAR,1.509619,3.0,0.629811,0.442661,41.251583,1.907941,30,monsoon
2,CHIRPS,1.157234,2.0,0.363293,0.238132,19.577975,1.325530,30,monsoon
3,MSWEP,3.650993,5.0,0.364182,0.870001,-86.418612,0.170735,30,monsoon
4,PERSIANN,0.784641,1.0,0.420690,0.190607,-13.296614,1.324466,30,monsoon


**Accuracy and CPI Calculation - Annual Scale**

In [129]:
df_merged2 = df_merged.copy()
df_merged2['date'] = pd.to_datetime(df_merged2['date'])
df_merged2['year'] = df_merged2['date'].dt.year
df_annual = df_merged2.groupby(by = ['year'])[['CHIRPS_prec',	'PERSIANN_prec', 'ERA5L_prec', 'prec_insitu', 'HAR_prec', 'MSWEP_prec']].sum().reset_index()
df_accuracy_annual = calculate_accuracy(df_annual)
df_accuracy_annual

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,ERA5L,2.809643,4.0,0.673415,0.795458,78.362257,1.897783,30
1,HAR,1.885047,3.0,0.646274,0.521707,50.496664,1.752727,30
2,CHIRPS,1.012259,2.0,0.423359,0.228232,18.599720,1.410547,30
3,MSWEP,3.903080,5.0,0.272525,0.864462,-85.913340,0.155534,30
4,PERSIANN,0.128746,1.0,0.621802,0.133549,-7.215277,1.347664,30


**Accuracy and CPI Calculation - Monthly Scale**

In [130]:
df_merged3 = df_merged.copy()
df_merged3['date'] = pd.to_datetime(df_merged3['date'])
df_merged3['year'] = df_merged3['date'].dt.year
df_merged3['month'] = df_merged3['date'].dt.month
df_monthly = df_merged3.groupby(by = ['year','month'])[['CHIRPS_prec',	'PERSIANN_prec', 'ERA5L_prec', 'prec_insitu', 'HAR_prec', 'MSWEP_prec']].sum().reset_index()
df_accuracy_monthly = calculate_accuracy(df_monthly)
df_accuracy_monthly

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,ERA5L,2.788451,4.0,0.980386,1.181780,78.362257,1.749925,360
1,HAR,1.362728,3.0,0.983823,0.786702,50.496664,1.436440,360
2,CHIRPS,0.960596,2.0,0.968697,0.403914,18.599720,1.201266,360
3,MSWEP,4.000000,5.0,0.964237,1.294782,-85.913340,0.136406,360
4,PERSIANN,0.880396,1.0,0.966579,0.402212,-7.215277,0.827587,360


**Calculation of Categorical Metrices for Dataset Evaluation**

In [131]:
import xarray as xr
import operator
import scores.categorical
df_merged

,date,stn_id,CHIRPS_prec,PERSIANN_prec,ERA5L_prec,HAR_prec,MSWEP_prec,prec_insitu
0,1990-01-01,102.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1990-01-01,103.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1990-01-01,104.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1990-01-01,105.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1990-01-01,107.0,0.0,0.2,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
905752,2019-12-31,1407.0,0.0,0.0,0.1,0.0,0.0,0.0
905753,2019-12-31,1408.0,0.0,0.0,0.0,0.0,0.0,0.0
905754,2019-12-31,1412.0,0.0,0.0,0.0,0.0,0.0,0.0
905755,2019-12-31,1416.0,0.0,0.0,0.0,0.0,0.0,0.0


In [170]:
#conversion to xarray
df_merged4 = df_merged.copy()

def calculate_categorical_metrics(dataframe_prec):
    df_categorical_metrics = pd.DataFrame()

    for dataset in datasets:
        dataset_categorical_metrics = {'dataset': dataset}

        obs_insitu = df_merged4.set_index(['stn_id', 'date'])['prec_insitu'].to_xarray()
        pred_dataset = df_merged4.set_index(['stn_id', 'date'])[f'{dataset}_prec'].to_xarray()

        #define event -  wet day is when precipitation is greater than 0.1
        event_operator = scores.categorical.ThresholdEventOperator(
            default_event_threshold=0.1, 
            default_op_fn=operator.ge)

        #conversion to binary
        dataset_binary, insitu_binary = event_operator.make_event_tables(pred_dataset, obs_insitu)
            
        #make contingency manager
        cm_dataset = event_operator.make_contingency_manager(dataset_binary, insitu_binary)
        
        #calculate metrics
        pod_dataset = cm_dataset.probability_of_detection()
        dataset_categorical_metrics['pod'] = pod_dataset.values
        far_dataset = cm_dataset.false_alarm_rate()
        dataset_categorical_metrics['far'] = far_dataset.values
        fb_dataset = cm_dataset.frequency_bias() 
        dataset_categorical_metrics['fb'] = fb_dataset.values
        csi_dataset = cm_dataset.critical_success_index()
        dataset_categorical_metrics['csi'] = csi_dataset.values
        
        # Wrap in a list to create a single-row DataFrame
        dataset_categorical_metrics = pd.DataFrame([dataset_categorical_metrics])
        df_categorical_metrics = pd.concat([df_categorical_metrics, dataset_categorical_metrics], ignore_index=True)

    return df_categorical_metrics

df_categorical_metrics = calculate_categorical_metrics(df_merged4)
df_categorical_metrics = df_categorical_metrics.round(2)
df_categorical_metrics


,dataset,pod,far,fb,csi
0,ERA5L,0.980492548272509,0.6714748405622892,2.676579151038876,0.3636724974881954
1,HAR,0.8537238792641543,0.3407086164126073,1.7143239864261475,0.45884329253659706
2,CHIRPS,0.33083784802841776,0.11698122101690521,0.6263221607033002,0.2553777338648836
3,MSWEP,0.6689041693953285,0.24295616273154558,1.2825901450655872,0.4145194167145769
4,PERSIANN,0.7628594391855689,0.36320568495854716,1.680285169970828,0.39785605613686464
